# 05 EBBA+ 样本检测与去除（Tier B，缩比）

EBBA+ 不使用任务相关样本，也不需要重训练。流程：

1. 用 EBBA 找到目标标签 $t$；
2. 收集“被模型判为 $t$”的图：一部分是真正的中毒图，一部分是目标类的干净图；
3. 对每张图施加 80 种损坏，算 $\mathrm{NTE}_k=\sum_j(e^{s'_k-s_k}-1)$；
4. 用 Eq.12 的 argmax 恢复原始标签；
5. 对 NTE 向量做 t-SNE + Birch 二聚类；论文没有说明无标签时如何认定中毒簇，
   本套件用“簇内平均目标 logit 相对最强非目标 logit 的领先幅度”来定向，并在 B07 报告敏感性；
6. 只把中毒簇重标为 NTE argmax，其余不动。

> 缩比协议：候选池 = 200 张中毒测试图 + 100 张目标类干净图（模型判为目标类的才进入流程）。
> 80 种损坏 = 19 类 × 4 强度 + 除以 2..5。所有结论为 proxy。
>
> **目标标签的处理**：EBBA+ 以 EBBA 检出的目标标签为前提。本 Notebook 用真实目标标签（oracle）
> 运行样本阶段，目的是单独检验 EBBA+ 本身；EBBA 能否找对目标属于 B02。若 B02 在某攻击上失败
> （例如 Blend），完整流水线在该攻击上就不可用，这一事实不会被掩盖。


In [ ]:
import json
import os
import sys
from pathlib import Path

# 定位仓库根目录：容器内是 /app，本地打开时可能是 notebooks/ 或其子目录。
ROOT = Path(os.environ.get("EBBA_ROOT", Path.cwd())).resolve()
if not (ROOT / "src" / "ebba").exists():
    for cand in [Path.cwd(), *Path.cwd().parents]:
        if (cand / "src" / "ebba").exists():
            ROOT = cand
            break
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("EBBA_ROOT", str(ROOT))

import matplotlib.pyplot as plt
import numpy as np

from ebba import config, nbutil
from ebba.claims import load_registry, run, run_many

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("project root:", ROOT)
print("artifacts   :", config.ARTIFACTS_DIR)
print("reports     :", config.REPORTS_DIR)
print("fast mode   :", config.FAST)


## B04：损坏是否让真值标签前移、目标标签后移？

这是 Lemma 2 的聚合版本：对 100 张中毒图 × 80 种损坏，统计
真值标签概率上升、目标标签概率下降、真值标签排名前移的比例。

判定分两层：方向层要求两类概率变化各自过半数（>0.60）；论文层（Fig.2 caption 的
“真值标签排名进入前 6 的比例超过 90%”）另记。排行榜统计在缩比模型上是退化的：
中毒图本来就把真值类排在目标类之后（平均排名 1.4，几乎没有前移空间），因此只作诊断。


In [ ]:
r = nbutil.show("B04", max_measured_chars=2600)
shift = r.measured["probability_shift"]
print("gt probability increase:", round(shift["gt_increase_fraction"], 4))
print("target probability drop:", round(shift["target_decrease_fraction"], 4))
print("gt rank forward         :", round(r.measured["rank_shift"]["gt_forward_fraction"], 4))


## B05：NTE argmax 能否恢复中毒图的原始标签？

对候选池中每张图算 Eq.11 的 NTE 向量，取 argmax 作为恢复标签。
中毒图应能以较高比例恢复为原类。对于源预测就是目标类的干净图，损坏后第 0 个分量的概率下降，
$\mathrm{NTE}_0$ 为负，argmax 不会回到 0，因此其恢复率接近 0——这正是需要聚类、而非只靠 argmax
区分两类图的原因，不是方法失败。

论文的 ResNet18 上恢复率接近其 PDR；缩比模型在 80 种损坏下的 top-1 精度远低于此，
所以本 Notebook 同时报告两件事：是否显著高于随机（机制方向），以及是否达到 0.85（论文级幅度）。
方向成立但幅度不足记 `PARTIAL`，接近随机才记 `CONTRADICTED`。


In [ ]:
r = nbutil.show("B05")
for attack, rec in r.measured["attacks"].items():
    print(f'{attack}: poisoned recovery={rec["recovery_accuracy_poisoned"]:.3f}, '
          f'clean-target acc={rec["accuracy_clean_target"]:.3f}, coverage={rec["coverage"]:.3f}')


## B06：干净目标类图的能量转移是否“随机”？

论文 Sec.4.2 说：干净图的能量会随机转移到其他标签，因此 **NTE 本身无法区分干净图与中毒图**，
必须再叠加 t-SNE + 聚类。这里检验干净目标类图的 NTE argmax 分布是否接近均匀
（卡方检验 p>0.01），并报告归一化熵与最大类占比；峰值 NTE 的均值比与 AUC 只作辅助诊断。


In [ ]:
r = nbutil.show("B06", max_measured_chars=2600)
for attack, rec in r.measured["attacks"].items():
    print(f'{attack}: entropy={rec["normalized_entropy"]:.2f}, max_share={rec["max_class_share"]:.2f}, '
          f'chi2_p={rec["chi2_p_value_vs_uniform"]:.3g}, random_like={rec["random_like"]}')


## B07：PDR / F1 是否达到可用量级？

完整流程（t-SNE + Birch）后的中毒检测精度。论文报告约 0.95；缩比模型只要求
PDR ≥ 0.70 且 F1 ≥ 0.70。这个门槛是有意放宽的：缩比模型与数据集都远小于论文。

**簇定向是套件的假设**：论文只写了 t-SNE + Birch，没有给出无标签时认定中毒簇的规则。
这里用簇内平均目标 margin（目标 logit − 最强非目标 logit）定向；同时报告两种敏感性：
沿用“峰值 NTE 更高者为中毒簇”的旧规则，以及用真实标签选最佳簇的上界。三者并列，
读者可以判断结论在多大程度上依赖这条假设。


In [ ]:
r = nbutil.show("B07")
for attack, rec in r.measured["attacks"].items():
    print(f'{attack}: PDR={rec["pdr"]:.3f}, precision={rec["precision"]:.3f}, '
          f'F1={rec["f1"]:.3f} (tp={rec["tp"]}, fp={rec["fp"]}, fn={rec["fn"]}) '
          f'[peak-rule PDR={rec["pdr_peak_rule"]:.3f}, oracle-cluster PDR={rec["pdr_best_cluster_oracle"]:.3f}]')


## B08：聚类方法是否影响结果？

对 HC、Birch、Mean Shift、DBSCAN 重复同一流程，要求四种方法的 PDR 极差 ≤ 0.10。
若某方法退化，代码回退到 KMeans 并记录 `method_used="fallback_kmeans"`。

论文 Table 5/7 的极差是 0.010，那是在 PDR≈0.95、两类簇几乎完全分开的设定下得到的。
缩比模型的 PDR 只有 0.7–0.85，簇本身重叠，因此换聚类方法会实际改变判为中毒的图。
若实测极差超限，它说明的是“缩比簇不纯”，而不是论文的四种方法在某数据集上真的差很多；
两种解读都在结果里给出（各方法的 PDR 逐个列出）。


In [ ]:
r = nbutil.show("B08", max_measured_chars=2600)
for attack, rec in r.measured["attacks"].items():
    print(attack, {m: round(v["pdr"], 3) for m, v in rec["methods"].items()},
          "spread=", round(rec["pdr_spread"], 3))


## B09 / B10：去除后门后的 BA 与 ASR

按论文做法，只改“被判为目标类且被聚类判为中毒”的图。指标分两层：

- **池内（条件指标）**：被 EBBA+ 检查的中毒图还有多少仍成功（ASR），干净目标类池的准确率；
- **全测试集（对应 Table 4）**：BA 降幅 ≈ 池内误报数 / 2,000；ASR_after ≈ 全局 ASR_before × 池内 ASR_after。

B09 的判定看方向（ASR 至少降一半）与全局 BA（降幅 ≤0.05）；论文级幅度（ASR 降 ≥0.80、BA 降 ≤0.01）
单独记录。池内 BA 降幅会明显更大，因为全部误报都集中在 200 张目标类图上——两个数字都展示，不取好看的那个。

B10 用全测试集 BA 检查 $\mathrm{BA_{after}}\ge\frac{K-1}{K}\mathrm{BA_{before}}$。


In [ ]:
r9 = nbutil.show("B09")
r10 = nbutil.show("B10")


## B13 / B14：多目标与实现敏感性

- **B13**：在实测能量向量上注入第二个与目标 0 等高的异常，检查 top-2 排序能否同时标出两者。
  这是统计量的机制检查，不是双目标攻击训练。
- **B14**：扫描 NTE 的温度 $T\in\{0.5,1,2,3\}$ 与截断 $cap\in\{\text{无},4,5\}$，
  以 $(T=1,cap=5)$ 为基准统计标签恢复一致率。它回答“Eq.9/10 没写温度是否要紧”。
  温度与截断分组报告：若只改 cap 时一致率恒为 1，而改温度时降到 0.90–0.96，
  说明歧义只出在温度上（与 A06 的合成实验结论一致）。


In [ ]:
r13 = nbutil.show("B13")
r14 = nbutil.show("B14", max_measured_chars=2600)


## 本 Notebook 的小结

- B04-B07 检验 EBBA+ 的核心机制：损坏使目标证据下降、NTE 能恢复原标签、聚类能分开两类；
- B08 检验聚类选择是否稳健；
- B09/B10 检验“只重标、不重训”能否同时降低 ASR 并保持 BA；
- B13/B14 是机制与实现敏感性检查，不是论文实验的复现。

任何一项失败都应先查 manifest 中的模型精度与攻击成功率，再判断是方法问题还是缩比设定问题；
NTE 恢复率本身没有门槛，它是被检验的结果（B05），不是构建前提。
